# Tester para utilizar los modelos
El notebook analiza ma MRI a estudiar, solicita los slices que se desea analizar y genera máscaras de segmentación automáticamente para delimitar el cerebro y la lesión isquémica. Posteriormente, muestra los datos volumétricos, un mapa 3D interactivo del órgano y lesión y, finalmente, genera un descargable para exportar las máscaras.

In [ ]:
# Dependencias opcionales si el entorno no las tiene
# %pip install numpy tensorflow nibabel matplotlib pillow plotly --quiet


In [ ]:
import os
from pathlib import Path
import zipfile

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model

# ---------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------
IMG_TARGET = 120
THRESHOLD_BRAIN = 0.5
THRESHOLD_ISQ = 0.8

MODEL_DIRS = [Path("models"), Path("models"), Path(".")]

MODEL_BRAIN_CANDIDATES = [
    "cerebro_segmentatio_model_RadImageNET.h5",
    "cerebro_segmentation_model_RadImageNET.h5",
    "cerebro_segmentatio_model_RadImageNET.keras",
    "cerebro_segmentation_model_RadImageNET.keras",
    "brain_unet_model.h5",         
    "cerebro_unet_model.h5",
]

MODEL_ISQ_CANDIDATES = [
    "isquemia_segmentatio_model_RadImageNET.h5",
    "isquemia_segmentation_model_RadImageNET.h5",
    "isquemia_segmentatio_model_RadImageNET.keras",
    "isquemia_segmentation_model_RadImageNET.keras",
    "isquemia_unet_model.h5",       
]

TEST_NIFTI = Path("test_dataset") / "permanent" / "Seq No.nii"

# Si los dejas en None, el notebook preguntará por teclado.
SLICE_INICIO = None
SLICE_FIN = None

# FUNCIONES AUXILIARES
def dice_coef(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )


def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


CUSTOM_OBJECTS = {
    "dice_coef": dice_coef,
    "dice_loss": dice_loss,
    "bce_dice_loss": bce_dice_loss,
}


def find_model_file(candidates, model_dirs=MODEL_DIRS):
    """Busca primero nombres exactos y luego hace búsqueda recursiva."""
    for model_dir in model_dirs:
        for filename in candidates:
            path = model_dir / filename
            if path.exists():
                return path

    # Búsqueda recursiva por si el usuario movió los modelos.
    for filename in candidates:
        matches = list(Path(".").rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        "No se encontró ningún modelo. Probé estos nombres:\n"
        + "\n".join(f"- {name}" for name in candidates)
        + "\n\nColoca los .h5/.keras en 'modelss/' o actualiza MODEL_*_CANDIDATES."
    )


def load_inference_model(path):
    """Carga modelos Keras/H5 para inferencia, sin recompilar el entrenamiento."""
    return load_model(path, custom_objects=CUSTOM_OBJECTS, compile=False)


def load_nifti_like_training(nifti_path):
    """Carga y normaliza el NIfTI replicando exactamente el preprocesado de Training.ipynb."""
    nii = nib.load(str(nifti_path))
    vol = nii.get_fdata()

    vol_min, vol_max = np.min(vol), np.max(vol)
    if vol_max > vol_min:
        vol = (vol - vol_min) / (vol_max - vol_min)
    else:
        vol = np.zeros_like(vol, dtype=np.float32)

    # IMPORTANTE: igual que en Training.ipynb
    vol = np.rot90(vol, k=3, axes=(0, 1))
    vol = np.flip(vol, axis=1)

    return nii, vol.astype(np.float32)


def resize_float_slice(slice_img, target=IMG_TARGET):
    """Redimensiona una slice float [0,1] al tamaño esperado por el modelo."""
    if slice_img.shape == (target, target):
        return slice_img.astype(np.float32)

    img_uint8 = np.clip(slice_img * 255, 0, 255).astype(np.uint8)
    resized = Image.fromarray(img_uint8).resize((target, target), Image.BILINEAR)
    return (np.asarray(resized).astype(np.float32) / 255.0)


def resize_binary_mask(mask, output_shape):
    """Devuelve la máscara al tamaño original de la slice usando vecino más cercano."""
    if mask.shape == tuple(output_shape):
        return mask.astype(np.uint8)

    resized = Image.fromarray(mask.astype(np.uint8) * 255).resize(
        (output_shape[1], output_shape[0]), Image.NEAREST
    )
    return (np.asarray(resized) > 127).astype(np.uint8)


def seleccionar_slices(vol, inicio=None, fin=None):
    total_slices = vol.shape[2]
    print(f"\nEl volumen tiene {total_slices} slices\n")

    cols = 7
    rows = int(np.ceil(total_slices / cols))
    plt.figure(figsize=(14, max(4, rows * 1.8)))
    for i in range(total_slices):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(vol[:, :, i], cmap="gray")
        plt.title(f"{i}")
        plt.axis("off")
    plt.suptitle("Todas las slices disponibles")
    plt.show()

    if inicio is not None and fin is not None:
        inicio, fin = int(inicio), int(fin)
        if inicio < 0 or fin >= total_slices or inicio > fin:
            raise ValueError(f"Rango inválido: {inicio} → {fin}. Debe estar entre 0 y {total_slices - 1}.")
        print(f"\nRango seleccionado: {inicio} → {fin}\n")
        return inicio, fin

    while True:
        try:
            inicio = int(input("Introduce SLICE INICIAL: "))
            fin = int(input("Introduce SLICE FINAL: "))

            if inicio < 0 or fin >= total_slices:
                print("Valores fuera de rango")
                continue
            if inicio > fin:
                print("El slice inicial no puede ser mayor que el final")
                continue
            break
        except ValueError:
            print("Debes introducir números enteros válidos")

    print(f"\nRango seleccionado: {inicio} → {fin}\n")
    return inicio, fin


def obtener_tamano_voxel(nii):
    return nii.header.get_zooms()[:3]


def calcular_volumenes(mask_brain_total, mask_isq_total, voxel_size):
    voxel_volume = voxel_size[0] * voxel_size[1] * voxel_size[2]  # mm³
    brain_voxels = np.sum(mask_brain_total)
    isq_voxels = np.sum(mask_isq_total)

    brain_volume_ml = (brain_voxels * voxel_volume) / 1000.0
    isq_volume_ml = (isq_voxels * voxel_volume) / 1000.0
    porcentaje = (isq_volume_ml / brain_volume_ml) * 100 if brain_volume_ml > 0 else 0

    return brain_volume_ml, isq_volume_ml, porcentaje


def mostrar_prediccion(slice_img, brain_mask, isq_mask, slice_index):
    plt.close("all")
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(slice_img, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(slice_img, cmap="gray")
    axes[1].imshow(brain_mask, cmap="Greens", alpha=0.4)
    axes[1].set_title("Cerebro")
    axes[1].axis("off")

    axes[2].imshow(slice_img, cmap="gray")
    axes[2].imshow(isq_mask, cmap="Reds", alpha=0.4)
    axes[2].set_title("Isquemia")
    axes[2].axis("off")

    axes[3].imshow(slice_img, cmap="gray")
    axes[3].imshow(brain_mask, cmap="Greens", alpha=0.4)
    axes[3].imshow(isq_mask, cmap="Reds", alpha=0.4)
    axes[3].set_title("Todo superpuesto")
    axes[3].axis("off")

    plt.suptitle(f"Slice {slice_index}")
    plt.show()


In [ ]:
# CARGA DE MODELOS Y DATOS

brain_model_path = find_model_file(MODEL_BRAIN_CANDIDATES)
isq_model_path = find_model_file(MODEL_ISQ_CANDIDATES)

print(f"Modelo cerebro:  {brain_model_path}")
print(f"Modelo isquemia: {isq_model_path}")

model_brain = load_inference_model(brain_model_path)
model_isq = load_inference_model(isq_model_path)
print("Modelos cargados correctamente")

nii, vol = load_nifti_like_training(TEST_NIFTI)
voxel_size = obtener_tamano_voxel(nii)
print(f"NIfTI cargado: {TEST_NIFTI}")
print(f"Forma tras preprocesado: {vol.shape}")
print(f"Tamaño de voxel: {voxel_size}")

# Inicializar máscaras con el tamaño real del volumen preprocesado.
h, w, d = vol.shape[:3]
mask_brain_total = np.zeros((h, w, d), dtype=np.uint8)
mask_isq_total = np.zeros((h, w, d), dtype=np.uint8)

slice_inicio, slice_fin = seleccionar_slices(vol, SLICE_INICIO, SLICE_FIN)
print(f"Procesando slices de {slice_inicio} a {slice_fin}")

for i in range(slice_inicio, slice_fin + 1):
    slice_img = np.squeeze(vol[:, :, i]).astype(np.float32)
    original_shape = slice_img.shape

    # Entrada al tamaño entrenado: (1, 120, 120, 1)
    slice_resized = resize_float_slice(slice_img, IMG_TARGET)
    input_img = slice_resized[np.newaxis, :, :, np.newaxis].astype(np.float32)

    # 1) Cerebro
    pred_brain = model_brain.predict(input_img, verbose=0)[0, ..., 0]
    mask_brain_target = (pred_brain > THRESHOLD_BRAIN).astype(np.uint8)

    # 2) Isquemia, usando la imagen limitada por la máscara cerebral, igual que el entrenamiento de isquemia
    input_img_brain = input_img * mask_brain_target[np.newaxis, :, :, np.newaxis]
    pred_isq = model_isq.predict(input_img_brain, verbose=0)[0, ..., 0]
    mask_isq_target = (pred_isq > THRESHOLD_ISQ).astype(np.uint8)

    # Guardar en el volumen completo con el tamaño original de la slice
    mask_brain_original = resize_binary_mask(mask_brain_target, original_shape)
    mask_isq_original = resize_binary_mask(mask_isq_target, original_shape)

    mask_brain_total[:, :, i] = mask_brain_original
    mask_isq_total[:, :, i] = mask_isq_original

    mostrar_prediccion(slice_img, mask_brain_original, mask_isq_original, i)

print("Predicción completada")


In [ ]:
brain_vol, isq_vol, porcentaje = calcular_volumenes(
    mask_brain_total,
    mask_isq_total,
    voxel_size
)

print("\n=============================")
print("RESULTADOS DEL ESTUDIO")
print("=============================")
print(f"Volumen de cerebro: {brain_vol:.2f} ml")
print(f"Volumen de isquemia: {isq_vol:.2f} ml")
print(f"Porcentaje afectado: {porcentaje:.2f} %")
print("=============================\n")


In [ ]:
# Visualización 3D interactiva
import plotly.graph_objects as go


def visualizar_3d_completo(mask_brain, mask_isq, nii, opacity_brain=0.1, opacity_isq=0.3):
    """Visualización 3D del cerebro e isquemia con proporciones reales."""
    dx, dy, dz = nii.header.get_zooms()[:3]
    print(f"Tamaño de voxel: X={dx}mm, Y={dy}mm, Z={dz}mm")

    brain_bool = mask_brain.astype(bool)
    isq_bool = mask_isq.astype(bool)

    X = np.arange(brain_bool.shape[0]) * dx
    Y = np.arange(brain_bool.shape[1]) * dy
    Z = np.arange(brain_bool.shape[2]) * dz

    Xf = X.repeat(brain_bool.shape[1] * brain_bool.shape[2])
    Yf = np.tile(Y.repeat(brain_bool.shape[2]), brain_bool.shape[0])
    Zf = np.tile(Z, brain_bool.shape[0] * brain_bool.shape[1])

    fig = go.Figure()

    fig.add_trace(go.Volume(
        x=Xf,
        y=Yf,
        z=Zf,
        value=brain_bool.flatten().astype(int),
        isomin=0.5,
        isomax=1,
        opacity=opacity_brain,
        surface_count=1,
        colorscale="Greens",
        name="Cerebro"
    ))

    fig.add_trace(go.Volume(
        x=Xf,
        y=Yf,
        z=Zf,
        value=isq_bool.flatten().astype(int),
        isomin=0.5,
        isomax=1,
        opacity=opacity_isq,
        surface_count=1,
        colorscale="Reds",
        name="Isquemia"
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode="data"
        ),
        width=900,
        height=700,
        title="Volumen 3D de cerebro e isquemia (escala real)"
    )

    fig.show()


visualizar_3d_completo(mask_brain_total, mask_isq_total, nii)


In [ ]:
def exportar_rois_compatible(mask_brain, mask_isq, slice_inicio, slice_fin, output_dir="output_roi"):
    """
    Exporta cada slice como imagen binaria 8-bit compatible con ImageJ.
    Luego comprime todo en un zip.
    """
    output_dir = Path(output_dir)
    brain_dir = output_dir / "cerebro"
    isq_dir = output_dir / "isquemia"
    brain_dir.mkdir(parents=True, exist_ok=True)
    isq_dir.mkdir(parents=True, exist_ok=True)

    for i in range(slice_inicio, slice_fin + 1):
        idx = str(i + 1).zfill(4)

        brain_slice = (mask_brain[:, :, i] * 255).astype("uint8")
        Image.fromarray(brain_slice).save(brain_dir / f"cerebro_slice_{idx}.tif")

        isq_slice = (mask_isq[:, :, i] * 255).astype("uint8")
        Image.fromarray(isq_slice).save(isq_dir / f"isquemia_slice_{idx}.tif")

    zip_path = output_dir / "rois_tif.zip"
    with zipfile.ZipFile(zip_path, "w") as zipf:
        for f in brain_dir.iterdir():
            zipf.write(f, arcname=f"cerebro/{f.name}")
        for f in isq_dir.iterdir():
            zipf.write(f, arcname=f"isquemia/{f.name}")

    print(f"Exportación completada. Zip generado en: {zip_path}")
    print("Para usar en ImageJ: abrir cada TIFF, aplicar Threshold y añadirlo al ROI Manager.")


exportar_rois_compatible(mask_brain_total, mask_isq_total, slice_inicio, slice_fin)
